<a href="https://colab.research.google.com/github/harshghugare80-lang/ML14/blob/main/Decision_Tree_Classification_Pruning_Overfitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Tree Classification

## Experiment

**Train and visualize a Decision Tree Classifier; analyze performance with and without pruning and depth control, and demonstrate overfitting using synthetic data.**

---

### Learning Objectives

After completing this experiment, the student will be able to:

1. Explain the working principle of a Decision Tree classifier.
2. Understand entropy, Gini impurity, information gain, and splitting.
3. Train and visualize a Decision Tree using scikit-learn.
4. Analyze the effect of tree depth on training and testing performance.
5. Apply pre-pruning using `max_depth` and other hyperparameters.
6. Understand post-pruning using cost-complexity pruning (`ccp_alpha`).
7. Demonstrate overfitting using a synthetic classification dataset.
8. Compare model performance using accuracy and classification metrics.


## 1. Introduction to Decision Trees

A **Decision Tree** is a supervised machine learning algorithm used for classification and regression. It represents decisions in a hierarchical tree structure. Each internal node contains a decision rule based on a feature, each branch represents an outcome of that rule, and each leaf node represents the final prediction.

For classification, the algorithm recursively partitions the dataset into more homogeneous groups. Common impurity measures are **Gini impurity** and **entropy**.

Important ML terminology:

- **Root Node:** The first and top-most decision node of the tree.
- **Internal Node:** A node where a feature-based decision is made.
- **Branch:** A path representing the outcome of a decision.
- **Leaf Node:** A terminal node containing the predicted class.
- **Gini Impurity:** Measures how mixed the classes are in a node.
- **Entropy:** Measures uncertainty or disorder in a node.
- **Information Gain:** Reduction in impurity after a split.
- **Maximum Depth:** Maximum number of levels allowed in the tree.
- **Pruning:** Removing unnecessary branches to reduce model complexity.
- **Overfitting:** When a model learns training-specific patterns/noise and performs poorly on unseen data.
- **Underfitting:** When a model is too simple to capture the underlying pattern.
- **Generalization:** The ability of a model to perform well on unseen data.


## 2. Problem Statement

**Train and visualize a Decision Tree classifier. Compare an unrestricted tree with trees controlled by `max_depth`, and analyze the effect of pruning using cost-complexity pruning. Finally, generate synthetic data to demonstrate how an excessively deep decision tree can overfit the training data. Evaluate the models using training and testing accuracy and classification metrics.**

The experiment emphasizes the relationship between **model complexity, bias, variance, pruning, depth control, and generalization performance**.


## 3. Decision Tree Working Principle

A Decision Tree selects a feature and threshold that produces a useful separation between classes.

For Gini impurity:

$$
Gini = 1 - \sum_{i=1}^{K} p_i^2
$$

where $p_i$ is the proportion of samples belonging to class $i$.

For entropy:

$$
Entropy = -\sum_{i=1}^{K} p_i \log_2(p_i)
$$

A good split generally produces child nodes with lower impurity. In this experiment, the default scikit-learn criterion is **Gini impurity**.

### Pre-pruning vs Post-pruning

**Pre-pruning** restricts tree growth during training using parameters such as:

- `max_depth`
- `min_samples_split`
- `min_samples_leaf`
- `max_leaf_nodes`

**Post-pruning** first allows the tree to grow and then removes branches that provide insufficient improvement. Scikit-learn provides **cost-complexity pruning** through `ccp_alpha`.


## 4. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (8, 5)


## 5. Load the Iris Dataset

The Iris dataset is a standard classification dataset containing measurements of iris flowers. It has three target classes.

We use the four numerical features:

- Sepal length
- Sepal width
- Petal length
- Petal width


In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

print("Feature names:")
print(iris.feature_names)

print("\nTarget names:")
print(iris.target_names)

print("\nDataset shape:", X.shape)

display(pd.concat([X, y], axis=1).head())


## 6. Basic Data Exploration

In [ ]:
print("Missing values:")
print(X.isnull().sum())

print("\nClass distribution:")
print(y.value_counts().sort_index())

print("\nStatistical summary:")
display(X.describe())


## 7. Train-Test Split

The dataset is divided into training and testing subsets. The model learns patterns from the training data, while the test data is kept separate to estimate performance on unseen observations.

`stratify=y` maintains approximately the same class proportions in both subsets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


## 8. Train an Unrestricted Decision Tree

First, we train a Decision Tree without explicitly controlling its maximum depth. This gives a baseline model and helps demonstrate how a tree can become unnecessarily complex.



In [ ]:
tree_unrestricted = DecisionTreeClassifier(
    criterion="gini",
    random_state=42
)

tree_unrestricted.fit(X_train, y_train)

y_train_pred = tree_unrestricted.predict(X_train)
y_test_pred = tree_unrestricted.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("Unrestricted Tree")
print("Tree depth      :", tree_unrestricted.get_depth())
print("Number of leaves:", tree_unrestricted.get_n_leaves())
print("Training accuracy:", round(train_acc, 4))
print("Testing accuracy :", round(test_acc, 4))


## 9. Visualize the Unrestricted Decision Tree

The following visualization shows the feature tests used by the tree. Each node displays the splitting condition, impurity, number of samples, and class information.


In [ ]:
plt.figure(figsize=(18, 9))

plot_tree(
    tree_unrestricted,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("Unrestricted Decision Tree - Iris Dataset")
plt.show()


## 10. Evaluate the Unrestricted Tree

In [ ]:
print("Classification Report - Unrestricted Tree")
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=iris.target_names
    )
)

cm = confusion_matrix(y_test, y_test_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
)

disp.plot()
plt.title("Confusion Matrix - Unrestricted Tree")
plt.show()


## 11. Decision Tree with Depth Control

`max_depth` is an important pre-pruning parameter. A smaller maximum depth restricts the number of sequential decisions the tree can make.

This can reduce variance and prevent the model from memorizing training-specific patterns. However, an excessively small depth can cause underfitting.


In [ ]:
depth_values = [1, 2, 3, 4, 5, 6, 8, 10, None]

depth_results = []

for depth in depth_values:
    model = DecisionTreeClassifier(
        criterion="gini",
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    depth_results.append({
        "max_depth": str(depth),
        "actual_depth": model.get_depth(),
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "leaves": model.get_n_leaves()
    })

depth_df = pd.DataFrame(depth_results)
display(depth_df)


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    depth_df["actual_depth"],
    depth_df["train_accuracy"],
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    depth_df["actual_depth"],
    depth_df["test_accuracy"],
    marker="s",
    label="Testing Accuracy"
)

plt.xlabel("Tree Depth")
plt.ylabel("Accuracy")
plt.title("Effect of Tree Depth on Training and Testing Accuracy")
plt.legend()
plt.grid(True)
plt.show()


### Interpretation

As tree depth increases, the tree receives more capacity to create detailed decision rules. Training accuracy generally increases because the model can fit increasingly complex patterns. Test accuracy may improve initially and then stabilize or decrease when the tree becomes unnecessarily complex.

This illustrates the **bias-variance trade-off**:

- Very shallow tree → high bias, possible underfitting.
- Appropriate depth → better generalization.
- Very deep tree → high variance, possible overfitting.


## 12. Select a Controlled-Depth Tree

For demonstration, we use `max_depth=3`. This produces a simpler and more interpretable model than an unrestricted tree.


In [ ]:

train_pred_depth3 = tree_depth3.predict(X_train)
test_pred_depth3 = tree_depth3.predict(X_test)

print("Controlled-Depth Tree")
print("Depth:", tree_depth3.get_depth())
print("Leaves:", tree_depth3.get_n_leaves())
print("Training accuracy:", round(accuracy_score(y_train, train_pred_depth3), 4))
print("Testing accuracy :", round(accuracy_score(y_test, test_pred_dtree_depth3 = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

tree_depth3.fit(X_train, y_train)
epth3), 4))


In [ ]:
plt.figure(figsize=(16, 8))

plot_tree(
    tree_depth3,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Pruned/Depth-Controlled Decision Tree - max_depth=3")
plt.show()


## 13. Cost-Complexity Pruning

Cost-complexity pruning provides a systematic post-pruning approach. The parameter `ccp_alpha` controls the complexity penalty.

- `ccp_alpha = 0` corresponds to no additional pruning.
- Larger values produce simpler trees.
- Excessive pruning can lead to underfitting.

We first obtain candidate effective alpha values from the training data.


In [ ]:
path = tree_unrestricted.cost_complexity_pruning_path(
    X_train,
    y_train
)

ccp_alphas = path.ccp_alphas

print("Number of candidate ccp_alpha values:", len(ccp_alphas))
print("First few values:", ccp_alphas[:10])


In [ ]:
pruning_results = []

# Use a manageable set of candidate alpha values.
candidate_alphas = np.unique(
    np.round(
        np.linspace(ccp_alphas.min(), ccp_alphas.max(), 12),
        6
    )
)

for alpha in candidate_alphas:
    model = DecisionTreeClassifier(
        criterion="gini",
        random_state=42,
        ccp_alpha=float(alpha)
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    pruning_results.append({
        "ccp_alpha": alpha,
        "depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred)
    })

pruning_df = pd.DataFrame(pruning_results)

display(pruning_df)


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    pruning_df["ccp_alpha"],
    pruning_df["train_accuracy"],
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    pruning_df["ccp_alpha"],
    pruning_df["test_accuracy"],
    marker="s",
    label="Testing Accuracy"
)

plt.xlabel("ccp_alpha")
plt.ylabel("Accuracy")
plt.title("Effect of Cost-Complexity Pruning")
plt.legend()
plt.grid(True)
plt.show()


## 14. Compare No Pruning, Depth Control and Pruning

In [ ]:
models = {
    "Unrestricted": tree_unrestricted,
    "Depth Controlled (3)": tree_depth3
}

# Choose the pruning candidate with the best test accuracy.
best_alpha_row = pruning_df.loc[
    pruning_df["test_accuracy"].idxmax()
]

best_alpha = float(best_alpha_row["ccp_alpha"])

tree_pruned = DecisionTreeClassifier(
    criterion="gini",
    random_state=42,
    ccp_alpha=best_alpha
)

tree_pruned.fit(X_train, y_train)
models[f"CCP Pruned ({best_alpha:.4f})"] = tree_pruned

comparison = []

for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    comparison.append({
        "Model": name,
        "Depth": model.get_depth(),
        "Leaves": model.get_n_leaves(),
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred, average="weighted"),
        "Recall": recall_score(y_test, test_pred, average="weighted"),
        "F1 Score": f1_score(y_test, test_pred, average="weighted")
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df.round(4))


In [ ]:
comparison_plot = comparison_df.set_index("Model")[
    ["Train Accuracy", "Test Accuracy"]
]

comparison_plot.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.ylabel("Accuracy")
plt.title("Training vs Testing Accuracy")
plt.xticks(rotation=15)
plt.ylim(0, 1.05)
plt.grid(axis="y")
plt.show()


## 15. Demonstrate Overfitting Using Synthetic Data

To make overfitting easier to observe, we create a synthetic classification dataset with:

- informative features,
- redundant features,
- noise,
- overlapping classes.

A very deep Decision Tree can create highly specific decision regions and memorize individual training observations. The resulting training accuracy may be much higher than the testing accuracy.

The **train-test accuracy gap** is a useful indicator for discussing overfitting.


In [ ]:
X_syn, y_syn = make_classification(
    n_samples=600,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=2,
    class_sep=0.8,
    flip_y=0.12,
    random_state=42
)

X_syn_train, X_syn_test, y_syn_train, y_syn_test = train_test_split(
    X_syn,
    y_syn,
    test_size=0.30,
    random_state=42,
    stratify=y_syn
)

print("Synthetic training samples:", X_syn_train.shape[0])
print("Synthetic testing samples :", X_syn_test.shape[0])


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_syn_train[:, 0],
    X_syn_train[:, 1],
    c=y_syn_train,
    alpha=0.65,
    edgecolor="k"
)

plt.xlabel("Synthetic Feature 1")
plt.ylabel("Synthetic Feature 2")
plt.title("Synthetic Training Data")
plt.show()


## 16. Train Shallow and Deep Trees on Synthetic Data

In [ ]:
synthetic_models = {
    "Shallow Tree (depth=2)": DecisionTreeClassifier(
        max_depth=2,
        random_state=42
    ),
    "Controlled Tree (depth=5)": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),
    "Deep Tree (unrestricted)": DecisionTreeClassifier(
        random_state=42
    )
}

synthetic_results = []

for name, model in synthetic_models.items():
    model.fit(X_syn_train, y_syn_train)

    train_pred = model.predict(X_syn_train)
    test_pred = model.predict(X_syn_test)

    synthetic_results.append({
        "Model": name,
        "Depth": model.get_depth(),
        "Leaves": model.get_n_leaves(),
        "Train Accuracy": accuracy_score(y_syn_train, train_pred),
        "Test Accuracy": accuracy_score(y_syn_test, test_pred),
        "Accuracy Gap": (
            accuracy_score(y_syn_train, train_pred)
            - accuracy_score(y_syn_test, test_pred)
        )
    })

synthetic_df = pd.DataFrame(synthetic_results)

display(synthetic_df.round(4))


In [ ]:
synthetic_df.set_index("Model")[
    ["Train Accuracy", "Test Accuracy"]
].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.ylabel("Accuracy")
plt.title("Synthetic Data: Shallow vs Deep Decision Trees")
plt.xticks(rotation=15)
plt.ylim(0, 1.05)
plt.grid(axis="y")
plt.show()


### Overfitting Observation

For a deep or unrestricted tree, it is common to observe:

**Very high training accuracy + lower testing accuracy**

This happens because the tree can form highly specific partitions around training observations. The model therefore has **high variance** and reduced generalization.

A controlled-depth tree usually has fewer leaves and simpler decision rules, which can improve generalization when the unrestricted tree is overfitting.


## 17. Visualize Decision Regions for the Synthetic Dataset

In [ ]:
def plot_decision_regions(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.25)
    plt.scatter(
        X[:, 0],
        X[:, 1],
        c=y,
        edgecolor="k",
        alpha=0.75
    )

    plt.xlabel("Synthetic Feature 1")
    plt.ylabel("Synthetic Feature 2")
    plt.title(title)
    plt.show()


plot_decision_regions(
    synthetic_models["Shallow Tree (depth=2)"],
    X_syn_test,
    y_syn_test,
    "Decision Regions - Shallow Tree"
)

plot_decision_regions(
    synthetic_models["Deep Tree (unrestricted)"],
    X_syn_test,
    y_syn_test,
    "Decision Regions - Deep Tree"
)


## 18. Final Performance Comparison

The following table summarizes the main observations from the Iris experiment and the synthetic overfitting experiment.

### Key points

1. An unrestricted tree is easy to train and can model complex relationships.
2. Increasing depth increases model capacity.
3. Depth control is a form of **pre-pruning**.
4. Cost-complexity pruning is a form of **post-pruning**.
5. Pruning can reduce unnecessary branches and improve interpretability.
6. A very deep tree may achieve excellent training accuracy but poorer test accuracy.
7. The goal is not maximum training accuracy; the goal is good **generalization**.
8. Model selection should consider unseen-data performance and complexity.


## 19. Result

A Decision Tree classifier was successfully trained and visualized on the Iris dataset. The experiment compared an unrestricted tree with depth-controlled and cost-complexity-pruned trees. The effect of `max_depth` and `ccp_alpha` on model complexity and accuracy was analyzed.

Using synthetic classification data, the experiment demonstrated that an unrestricted/deep Decision Tree can achieve very high training accuracy while producing a larger training-testing performance gap. This demonstrates the concept of **overfitting**. Depth control and pruning provide practical mechanisms for reducing model complexity and improving generalization.


## 20. Conclusion

Decision Trees are interpretable supervised learning models that make predictions through a sequence of feature-based decisions. However, unrestricted tree growth can increase variance and cause overfitting. The experiment demonstrated that **pre-pruning using depth control** and **post-pruning using cost-complexity pruning** can control model complexity.

Therefore, a suitable tree should balance **fit, complexity, interpretability, and generalization performance** rather than simply maximizing training accuracy.


## 21. Viva Questions

1. What is a Decision Tree classifier?
2. What is the role of the root node?
3. What is a leaf node?
4. What is Gini impurity?
5. What is entropy?
6. What is information gain?
7. Why can Decision Trees overfit?
8. What does `max_depth` control?
9. What is pre-pruning?
10. What is post-pruning?
11. What is `ccp_alpha`?
12. Why is a train-test split required?
13. What is the bias-variance trade-off?
14. How can a shallow tree underfit the data?
15. Why can an unrestricted tree have high training accuracy?
16. What is the difference between training and testing accuracy?
17. Why is model complexity important?
18. How does pruning affect interpretability?
19. What does generalization mean?
20. Why should testing performance be considered when selecting tree depth?


## 22. Important Terms for Journal Writing

**Decision Tree, Classification, Supervised Learning, Root Node, Internal Node, Leaf Node, Branch, Gini Impurity, Entropy, Information Gain, Splitting Criterion, Tree Depth, Maximum Depth, Pre-pruning, Post-pruning, Cost-Complexity Pruning, `ccp_alpha`, Overfitting, Underfitting, Bias, Variance, Generalization, Training Accuracy, Testing Accuracy, Precision, Recall, F1-score, Confusion Matrix, Hyperparameter, Model Complexity, Decision Boundary.**
